In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time

BASE_URL   = "https://www.emploi.ma"
SEARCH_URL = "https://www.emploi.ma/recherche-jobs-maroc/{term}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

search_terms = [
    "data engineer", "data analyst", "data scientist",
    "business intelligence", "power bi", "ETL", "big data",
    "data warehouse", "ingénieur data", "analyste données",
    "devops", "cloud engineer", "ingénieur cloud",
    "ingénieur devops", "kubernetes", "terraform",
    "machine learning", "intelligence artificielle",
    "deep learning", "NLP", "MLOps",
    "développeur full stack", "développeur backend",
    "développeur frontend", "ingénieur logiciel",
    "développeur java", "développeur python",
    "développeur .net", "développeur react",
    "développeur angular", "développeur node.js",
    "développeur php", "laravel", "symfony",
    "développeur web", "wordpress",
    "développeur mobile", "android", "flutter", "react native",
    "architecte logiciel", "solution architect",
    "technical lead", "lead développeur",
    "chef de projet IT", "directeur informatique",
    "ingénieur cybersécurité", "sécurité informatique",
    "analyste sécurité", "RSSI", "SOC analyst",
    "ingénieur réseaux", "administrateur systèmes",
    "ingénieur systèmes", "administrateur base de données",
    "ingénieur télécoms", "VoIP",
    "consultant SAP", "SAP ABAP", "oracle ERP",
    "salesforce", "dynamics 365", "servicenow",
    "consultant fonctionnel", "AMOA",
    "power platform", "sharepoint", "azure administrator",
    "ingénieur QA", "test automation", "testeur logiciel",
    "support informatique", "technicien helpdesk",
    "technicien informatique", "technicien réseaux",
    "product manager", "product owner", "scrum master",
    "business analyst", "analyste fonctionnel",
    "systèmes embarqués", "firmware", "SCADA",
    "ingénieur automatisme", "IoT",
    "auditeur informatique", "ISO 27001", "DPO",
    "ingénieur informatique", "développeur",
    "ingénieur développement", "responsable informatique",
    "DSI", "architecte cloud",
]


def scrape_emploi_ma(term, max_pages=10):
    jobs = []
    url  = SEARCH_URL.format(term=term.replace(" ", "%20"))
    seen_urls = set()  # ← track seen job URLs

    for page in range(1, max_pages + 1):
        page_url = url if page == 1 else f"{url}?p={page}"
        print(f"  Page {page} — '{term}'...")

        try:
            res  = requests.get(page_url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(res.text, "html.parser")
            job_cards = soup.select(".card-job")

            if not job_cards:
                print("    → No more jobs.")
                break

            new_jobs = 0
            for card in job_cards:
                try:
                    title_tag   = card.select_one("h2 a") or card.select_one("a")
                    title       = title_tag.get_text(strip=True) if title_tag else ""
                    href        = title_tag.get("href", "") if title_tag else ""
                    job_url     = BASE_URL + href if href.startswith("/") else href

                    # ── Stop if we've seen this URL before ──
                    if job_url in seen_urls:
                        continue
                    seen_urls.add(job_url)
                    new_jobs += 1

                    company_tag  = card.select_one(".company-name")
                    company      = company_tag.get_text(strip=True) if company_tag else ""
                    details      = card.select(".card-job-detail")
                    detail_texts = [d.get_text(strip=True) for d in details]
                    desc_tag     = card.select_one(".card-job-description")
                    description  = desc_tag.get_text(strip=True) if desc_tag else ""
                    date_tag     = card.select_one("time") or card.select_one(".date")
                    date_posted  = date_tag.get("datetime") or date_tag.get_text(strip=True) if date_tag else ""

                    jobs.append({
                        "source":      "emploi.ma",
                        "title":       title,
                        "company":     company,
                        "detail_1":    detail_texts[0] if len(detail_texts) > 0 else "",
                        "detail_2":    detail_texts[1] if len(detail_texts) > 1 else "",
                        "detail_3":    detail_texts[2] if len(detail_texts) > 2 else "",
                        "detail_4":    detail_texts[3] if len(detail_texts) > 3 else "",
                        "description": description,
                        "date_posted": date_posted,
                        "job_url":     job_url,
                        "search_term": term,
                        "scraped_at":  datetime.now().isoformat(),
                    })

                except Exception as e:
                    print(f"    → Card error: {e}")
                    continue

            print(f"    → {new_jobs} new jobs on page {page}")

            # ── If no new jobs were found, we hit the last page ──
            if new_jobs == 0:
                print("    → Duplicate page detected, stopping.")
                break

            time.sleep(1.5)

        except Exception as e:
            print(f"  → Request failed: {e}")
            break

    return jobs

# ── Run ───────────────────────────────────────────────────────────────────────
all_jobs = []

for term in search_terms:
    results = scrape_emploi_ma(term=term, max_pages=10)
    all_jobs.extend(results)
    print(f"  → Cumulative total: {len(all_jobs)}\n")

df = pd.DataFrame(all_jobs)

before = len(df)
df = df.drop_duplicates(subset=["job_url"])
print(f"Deduplication: {before} → {len(df)} unique jobs")

def clean_text(val):
    if isinstance(val, str):
        val = val.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
        val = ' '.join(val.split())
    return val

df = df.apply(lambda col: col.map(clean_text))

df.to_csv(
    "../data/jobs_emploi_ma.tsv",
    sep="\t",
    index=False,
    encoding="utf-8",
    lineterminator="\n"
)

print(f"Saved {len(df)} jobs to jobs_emploi_ma.tsv")

  Page 1 — 'data engineer'...
    → 6 new jobs on page 1
  Page 2 — 'data engineer'...
    → 0 new jobs on page 2
    → Duplicate page detected, stopping.
  → Cumulative total: 6

  Page 1 — 'data analyst'...
    → 2 new jobs on page 1
  Page 2 — 'data analyst'...
    → 0 new jobs on page 2
    → Duplicate page detected, stopping.
  → Cumulative total: 8

  Page 1 — 'data scientist'...
    → 1 new jobs on page 1
  Page 2 — 'data scientist'...
    → 0 new jobs on page 2
    → Duplicate page detected, stopping.
  → Cumulative total: 9

  Page 1 — 'business intelligence'...
    → 2 new jobs on page 1


In [ ]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

url = "https://www.emploi.ma/recherche-jobs-maroc/data%20engineer"
res = requests.get(url, headers=HEADERS, timeout=15)

print(f"Status: {res.status_code}")
print(f"Page size: {len(res.text)} chars")

# Save HTML to file so you can open it in browser
with open("emploi_debug.html", "w", encoding="utf-8") as f:
    f.write(res.text)

print("Saved to emploi_debug.html")

# Try to find any job-like elements
soup = BeautifulSoup(res.text, "html.parser")

# Print all unique class names to find job card classes
all_classes = set()
for tag in soup.find_all(True):
    for c in tag.get("class", []):
        all_classes.add(c)

print("\nAll CSS classes found on page:")
for c in sorted(all_classes):
    print(f"  .{c}")

In [ ]:
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "fr-FR,fr;q=0.9",
}

# Test different pagination formats
test_urls = [
    "https://www.emploi.ma/recherche-jobs-maroc/développeur?page=2",
    "https://www.emploi.ma/recherche-jobs-maroc/développeur?p=2",
    "https://www.emploi.ma/recherche-jobs-maroc/développeur/2",
    "https://www.emploi.ma/recherche-jobs-maroc/développeur?start=15",
    "https://www.emploi.ma/recherche-jobs-maroc/développeur?from=15",
]

for url in test_urls:
    res  = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(res.text, "html.parser")
    jobs = soup.select(".card-job")
    print(f"{len(jobs)} jobs → {url}")